In [1]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import datetime as dt
import matplotlib.pyplot as plt
import os

In [2]:
import sys
from pathlib import Path

# Add project root (parent of the optimization folder) to sys.path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [3]:
THEATER_NAME = "AMC Boston Common 19"
NUM_SCREENS = 19
SCREENS = [f"Screen_{i}" for i in range(1, NUM_SCREENS+1)]

# Optimization Formulation

1. Parameters

- $A_{i,d}$: base expected attendance for movie i per day d (before adjustments)
- $P$: ticket price
- $P_0$: base ticket price (reference)
- $\epsilon$: price elasticity of demand
- $D_i$: duration of movie i in minutes
- $G_i$: genres of movie i
- $B$: buffer time between shows
- $C_j$: capacity of screen j
- $H_{j,d}$: hours screen j operates on day d 
- $K$: maximum times a movie can be shown on a given day
- $L$: min number of genres per day
- $\rho$: intra-day repetition penalty (cost per showing beyond first per day)

**Demand Adjustments:**
- Price adjustment: $A'_{i,d} = A_{i,d} \times \left(\frac{P_0}{P}\right)^\epsilon$
  - Higher prices reduce demand (realistic elasticity)
  - $\epsilon = 1.2$ means 1% price increase → 1.2% demand decrease

2. Decision Variables:

- $s_{i,d} \in \{0, 1\}$: whether movie i is scheduled on day d
- $x_{i, d, j} \geq 0$: number of times we show movie i on screen j on day d (integer)
- $r_{i, d} \geq 0$: realized tickets served for movie i on day d (continuous)
- $g_{genre, d} \in \{0, 1\}$: whether genre is represented on day d

3. Objective function: Maximize total revenue with repetition penalty:

$$\max \left( \sum_{i, d} P \times r_{i,d} - \text{Repetition Penalty} \right)$$

Where:
$$\text{Repetition Penalty} = \rho \times \left(\sum_{i,d,j} x_{i,d,j} - \sum_{i,d} s_{i,d}\right)$$

- The penalty equals: cost × (total showings - number of scheduled movie-days)
- This penalizes each showing beyond the first per movie per day
- Example: If a movie is shown 3 times on Monday, penalty = $\rho \times (3 - 1) = 2\rho$
- Discourages over-scheduling the same movie on the same day
- Represents customer dissatisfaction from repetitive programming

4. Constraints
- Total screen time: $\sum_{i} x_{i,d,j} (D_i+B) \leq H_{d,j} \quad \forall d,j$
- Tickets served won't exceed capacity:  $r_{i,d}\leq \sum_{j} x_{i, d,j}\cdot C_j \quad \forall i,d$
- Can't sell more tickets than adjusted demand: $r_{i,d} \leq A'_{i,d} \quad \forall i,d$
- If not scheduled, no showings:  $\sum_{j} x_{i, d,j} \leq K \times s_{i,d} \quad \forall i,d$
- Each movie shown at most K times: $\sum_{j} x_{i, d,j} \leq K \quad \forall i,d$
- Genre linking (upper): $g_{genre,d} \leq \sum_{i \in I_{genre}} s_{i,d} \quad \forall genre, d$
- Genre linking (lower): $s_{i,d} \leq g_{genre,d} \quad \forall i \in I_{genre}, d$
- Minimum genres per day: $\sum_{genre} g_{genre, d} \geq L \quad \forall d$


### Enhanced Objective Function with Realistic Tradeoffs

The optimization model includes two key enhancements that make sensitivity analysis more meaningful:

#### 1. Price-Demand Elasticity

**Problem:** Original model had ticket price always increasing revenue (unrealistic)

**Solution:** Demand now responds to price changes:
- Formula: `adjusted_demand = base_demand × (base_price / current_price)^elasticity`
- With elasticity = 1.2: A 10% price increase → 12% demand decrease
- Creates realistic tradeoff: Higher prices mean more revenue per ticket but fewer tickets sold

**Impact on sensitivity analysis:**
- Price sensitivity now shows an optimal price point (not just "higher is better")
- Captures diminishing returns from price increases

#### 2. Intra-Day Repetition Penalty

**Problem:** Without penalty, model may over-schedule the same movie multiple times per day, leading to:
- Customer dissatisfaction from lack of variety
- Reduced appeal for repeat visitors
- Inefficient use of screen time

**Solution:** Added repetition penalty to objective function:
- Formula: `penalty = cost_per_showing × (total_showings - scheduled_movie_days)`
- Default: $50 cost per showing beyond the first per movie per day
- Example: Movie shown 4 times on Friday → penalty = $50 × (4 - 1) = $150
- Represents opportunity cost of not showing different content

**Impact on sensitivity analysis:**
- Creates tradeoff between maximizing immediate revenue and maintaining programming variety
- Higher penalty → more diverse daily schedules with fewer repeat showings
- Lower penalty → model may concentrate on highest-demand movies
- Balances short-term revenue with customer satisfaction

#### Configurable Parameters

```python
BASE_TICKET_PRICE = 11.31               # Reference price
PRICE_ELASTICITY = 1.2                  # Demand elasticity
REPETITION_PENALTY_PER_SHOWING = 50     # Dollar cost per extra showing per day
MIN_GENRES = 5                          # Minimum genres required per day
```

These parameters can be tuned based on empirical data or used in sensitivity analysis to understand their impact.

### Genre Diversity Constraint Implementation

The genre diversity constraint ensures that each day features at least `MIN_GENRES` different genres, providing variety for moviegoers.

**Implementation approach:**
- **Genre-day binary variables:** `g[genre, d]` = 1 if genre is represented on day d, 0 otherwise
- **Linking constraints:** 
  - If any movie with a genre is scheduled, `g[genre, d] = 1`
  - If no movies with a genre are scheduled, `g[genre, d] = 0`
- **Diversity constraint:** $\sum_{genres} g[genre, d] \geq MIN\_GENRES$ for all days d

**Efficiency:** This approach adds only 19 genres × 7 days = 133 binary variables, avoiding the need for movie-genre-day combinations which would create thousands of variables.

**Handling multi-genre movies:** Since movies can have multiple genres, the linking constraints automatically handle this by setting `g[genre, d] = 1` for all genres that the scheduled movie contains.

**Difference from repetition penalty:** While the repetition penalty discourages showing the same movie multiple times per day, the genre diversity constraint ensures a minimum variety of genres across different movies each day.


# Preprocessing

### Parameters

In [4]:
DAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
MIN_GENRES = 5
OPENING_TIME = dt.time(11,0)
CLOSING_TIME=dt.time(23,0)
TICKET_PRICE = 15

# Price elasticity parameters
BASE_TICKET_PRICE = 11.31 # Reference price for demand calculations
PRICE_ELASTICITY = 1.1 # Demand decreases by 1.1% for every 1% price increase
REPETITION_PENALTY_PER_SHOWING = 50  # Dollar cost per showing beyond first on same day

TICKET_PRICES = [TICKET_PRICE]*len(DAYS)
today = dt.date.today()
open_dt = dt.datetime.combine(today, OPENING_TIME)
close_dt = dt.datetime.combine(today, CLOSING_TIME)
delta = close_dt - open_dt


OPERATING_MIN_PER_DAY = {
    day: delta.total_seconds()/60 for day in DAYS  # same for all days for now
}

SCREEN_CAPACITIES = {s: 210 
for s in SCREENS[:19]}
SCREEN_CAPACITIES["Screen_19"] = 600

BUFFER_MIN = 15

MAX_SHOWINGS_PER_MOVIE_PER_DAY = 6

In [5]:
def preprocess_genre_data(movie_genre_df):
    """
    Preprocess genre data into a dictionary mapping genres to movies.
    
    Args:
        movie_genre_df: DataFrame with movie IDs as index and genre columns as binary indicators
        movie_ids: List of movie IDs to include
    
    Returns:
        tuple: (genre_to_movies dict, genre_columns list)
            - genre_to_movies: Maps genre -> list of movie IDs that have that genre
            - genre_columns: List of all genre names
    """
    genre_columns = ['action', 'adventure', 'animation', 'comedy', 'crime', 
                    'documentary', 'drama', 'family', 'fantasy', 'history', 
                    'horror', 'music', 'mystery', 'romance', 'science_fiction', 
                    'thriller', 'tv_movie', 'war', 'western']
    
    genre_to_movies = {}
    for genre in genre_columns:
        # Filter movies that have this genre
        subset = movie_genre_df[movie_genre_df[genre] == 1]
        
        # Check if subset is not empty (using .empty instead of boolean check on DataFrame)
        if not subset.empty:
            # Get the movie IDs from the 'id' column
            subset_ids = subset['id'].tolist()
            # subset_ids = [i for i in subset_ids if i != '978']
            genre_to_movies[genre] = [str(movie_id) for movie_id in subset_ids]
        else:
            genre_to_movies[genre] = []
    
    return genre_to_movies, genre_columns

print("✓ Genre preprocessing function created: preprocess_genre_data()")
print("\nThis function converts genre data from a DataFrame to a dictionary mapping genres to movie IDs.")


✓ Genre preprocessing function created: preprocess_genre_data()

This function converts genre data from a DataFrame to a dictionary mapping genres to movie IDs.


In [6]:
current = pd.read_csv("inputs/recent_demand_predictions_all_theatres.csv")

In [7]:
final_merged = pd.read_csv('../data/cleaned/final_merged_dataset_with_genres.csv')

In [8]:
current_ids = current.id.to_list()

In [9]:
final_merged = final_merged.loc[current_ids]

In [11]:
# Fixed: Create proper demand structure with (movie_id, day) tuples
# and distribute weekly demand across days

# ============================================================================
# TODO: INTEGRATION WITH CHOICE MODEL
# ============================================================================
# To use choice model predictions instead of historical gross data:
# 
# Option 1: Load from Python file (recommended - easiest)
# from predictions.results.choice_model_for_optimization import demand, runtimes, movie_ids
#   # That's it! The choice model already formats data in the exact format needed.
#   # Then skip the code below (comment out the for loop)

import json
with open('inputs/choice_model_demand.json', 'r') as f:
    demand_json = json.load(f)
    # Convert string keys back to tuple keys: "123_Mon" -> (123, "Mon")

    demand = {tuple(k.split('_')): v for k, v in demand_json.items()}
    # demand = {tuple(k.split('_')): v for k, v in demand_json.items() if k.split('_')[0]!="978"}

with open('inputs/choice_model_runtimes.json', 'r') as f:
    runtimes = json.load(f)
    # runtimes = {k:v for k,v in runtimes.items() if k!=978}

with open('inputs/choice_model_metadata.csv', 'r') as f:
    metadata = pd.read_csv(f)
    movie_ids = metadata['movie_id'].astype(str).tolist()
    # movie_ids.remove("978")

In [12]:
demand

{('978', 'Mon'): 692515,
 ('978', 'Tue'): 692515,
 ('978', 'Wed'): 779080,
 ('978', 'Thu'): 865644,
 ('978', 'Fri'): 1125338,
 ('978', 'Sat'): 1385031,
 ('978', 'Sun'): 1385031,
 ('3664', 'Mon'): 88489,
 ('3664', 'Tue'): 88489,
 ('3664', 'Wed'): 99550,
 ('3664', 'Thu'): 110611,
 ('3664', 'Fri'): 143794,
 ('3664', 'Sat'): 176978,
 ('3664', 'Sun'): 176978,
 ('3488', 'Mon'): 65479,
 ('3488', 'Tue'): 65479,
 ('3488', 'Wed'): 73663,
 ('3488', 'Thu'): 81848,
 ('3488', 'Fri'): 106403,
 ('3488', 'Sat'): 130958,
 ('3488', 'Sun'): 130958,
 ('2464', 'Mon'): 31167,
 ('2464', 'Tue'): 31167,
 ('2464', 'Wed'): 35063,
 ('2464', 'Thu'): 38959,
 ('2464', 'Fri'): 50647,
 ('2464', 'Sat'): 62335,
 ('2464', 'Sun'): 62335,
 ('2876', 'Mon'): 31104,
 ('2876', 'Tue'): 31104,
 ('2876', 'Wed'): 34992,
 ('2876', 'Thu'): 38881,
 ('2876', 'Fri'): 50545,
 ('2876', 'Sat'): 62209,
 ('2876', 'Sun'): 62209,
 ('2486', 'Mon'): 30247,
 ('2486', 'Tue'): 30247,
 ('2486', 'Wed'): 34028,
 ('2486', 'Thu'): 37809,
 ('2486', 'Fri'

In [13]:
demand = {tuple(k.split('_')): 80000 if k.split('_')[0] == "978" else v for k, v in demand_json.items()}

In [14]:
demand = {k: v / 500 for k,v in demand.items()}

In [19]:
def createModel(runtimes, demand, ticket_price, max_showings_per_day, operating_min_per_day, 
                screen_capacity, movie_ids, days, screens, buffer_min, 
                genre_to_movies=None, genre_columns=None, min_genres=None, 
                base_price=None, price_elasticity=None, repetition_penalty_per_showing=None,
                verbose=False):
    
    m = gp.Model("CinemaShowtimeScheduling")
    if not verbose:
        m.Params.OutputFlag = 0  # silence output by default
    
    adjusted_demand = {}
    if base_price is not None and price_elasticity is not None and base_price > 0:
        # Demand adjustment: demand_new = demand_base * (base_price / current_price)^elasticity
        price_ratio = base_price / ticket_price
        demand_multiplier = price_ratio ** price_elasticity
        for key, val in demand.items():
            adjusted_demand[key] = val * demand_multiplier
        if verbose:
            print(f"Price elasticity applied: {demand_multiplier:.3f}x demand adjustment")
    else:
        adjusted_demand = demand.copy()
    
    # Decision variables
    x = m.addVars(
        movie_ids, days, screens,
        vtype=GRB.INTEGER,
        lb=0,
        name="x"
    )
    
    # r[i,d] >= 0 continuous - realized tickets
    r = m.addVars(
        movie_ids, days,
        vtype=GRB.CONTINUOUS,
        lb=0.0,
        name="r"
    )
    
    # s[i,d] binary - whether movie is scheduled
    s = m.addVars(
        movie_ids, days,
        vtype=GRB.BINARY,
        name="s"
    )

    g = m.addVars(genre_columns, days, vtype=GRB.BINARY, name="genre")

    # Objective: maximize total revenue from ticket sales
    revenue_from_tickets = gp.quicksum(ticket_price * r[i, d] for i in movie_ids for d in days)
    
    # Add genre diversity bonus if enabled
    # This represents additional customer satisfaction and repeat business
    objective_expr = revenue_from_tickets

    if repetition_penalty_per_showing is not None and repetition_penalty_per_showing > 0:
        # Penalty = cost × (total_showings - number_of_scheduled_movie_days)
        # This penalizes each showing beyond the first per movie-day
        # E.g., if movie shown 3 times on Monday: penalty = cost × (3 - 1) = 2×cost
        
        total_showings = gp.quicksum(x[i, d, j] for i in movie_ids for d in days for j in screens)
        scheduled_movie_days = gp.quicksum(s[i, d] for i in movie_ids for d in days)
        
        repetition_penalty = repetition_penalty_per_showing * (total_showings - scheduled_movie_days)
        
        objective_expr = objective_expr - repetition_penalty
        
        if verbose:
            print(f"Intra-day repetition penalty: ${repetition_penalty_per_showing:.0f} per showing beyond first per day")
    
    m.setObjective(objective_expr, GRB.MAXIMIZE)
    
    # 1) Screen time constraint per screen & day
    for d in days:
        for j in screens:
            m.addConstr(
                gp.quicksum(x[i, d, j] * (runtimes[i] + buffer_min) for i in movie_ids)
                <= operating_min_per_day[d],
                name=f"Time_{d}_{j}"
            )
    
    # 2) Capacity: r[i,d] <= sum_j x[i,d,j] * capacity_j
    for i in movie_ids:
        for d in days:
            m.addConstr(
                r[i, d] <= gp.quicksum(x[i, d, j] * screen_capacity[j] for j in screens),
                name=f"Capacity_{i}_{d}"
            )
    
    # 3) Demand: r[i,d] <= adjusted_demand[i,d]
    for i in movie_ids:
        for d in days:
            demand_val = adjusted_demand.get((i, d), 0)
            m.addConstr(
                r[i, d] <= demand_val,
                name=f"Demand_{i}_{d}"
            )
    
    # 4) Frequency + schedule linking
    for i in movie_ids:
        for d in days:
            m.addConstr(
                gp.quicksum(x[i, d, j] for j in screens)
                <= max_showings_per_day * s[i, d],
                name=f"FreqLimit_{i}_{d}"
            )
    
    # 5) Genre diversity constraints (if enabled)


        # Link genre variables to movie scheduling
    for genre in genre_columns:
        for d in days:
            # Get movies that have this genre (already preprocessed)
            movies_with_genre = genre_to_movies.get(genre, [])
            
            if movies_with_genre:
                # If no movies with this genre are scheduled, g[genre,d] must be 0
                m.addConstr(
                    g[genre, d] <= gp.quicksum(s[i, d] for i in movies_with_genre),
                    name=f"GenreUpper_{genre}_{d}"
                )
                
                # If any movie with this genre is scheduled, g[genre,d] must be 1
                for i in movies_with_genre:
                    m.addConstr(
                        s[i, d] <= g[genre, d],
                        name=f"GenreLower_{genre}_{d}_{i}"
                    )
    
    # Add diversity constraint: at least min_genres different genres per day
    if min_genres is not None and min_genres > 0:
        for d in days:
            m.addConstr(
                gp.quicksum(g[genre, d] for genre in genre_columns) >= min_genres,
                name=f"MinGenres_{d}"
            )
    
    # Return model AND variables so we can access them later
    return m, x, r, s, g


In [20]:
# Preprocess genre data
# current  = current[current['id']!=978]
genre_to_movies, genre_columns = preprocess_genre_data(current)

# Create model and get variable references
model, x, r, s, g = createModel(
    runtimes, demand, TICKET_PRICE, MAX_SHOWINGS_PER_MOVIE_PER_DAY, 
    OPERATING_MIN_PER_DAY, SCREEN_CAPACITIES, movie_ids, DAYS, SCREENS, 
    BUFFER_MIN, 
    genre_to_movies=genre_to_movies,
    genre_columns=genre_columns,
    min_genres=MIN_GENRES,
    base_price=BASE_TICKET_PRICE,
    price_elasticity=PRICE_ELASTICITY,
    repetition_penalty_per_showing=REPETITION_PENALTY_PER_SHOWING,
    verbose=True
)

print(f"\nModel created with {len(movie_ids)} movies, {len(DAYS)} days, {len(SCREENS)} screens")
print(f"Genre diversity constraint: MIN_GENRES = {MIN_GENRES}")
print(f"Price elasticity: {PRICE_ELASTICITY}")
print(f"Total decision variables: {model.getAttr('NumVars')}")
print(f"Total constraints: {model.getAttr('NumConstrs')}")


Price elasticity applied: 0.733x demand adjustment
Intra-day repetition penalty: $50 per showing beyond first per day

Model created with 15 movies, 7 days, 19 screens
Genre diversity constraint: MIN_GENRES = 5
Price elasticity: 1.1
Total decision variables: 0
Total constraints: 0


In [21]:
# Test: Run optimization directly to check model
model.optimize()

print(f"\nOptimal objective value: ${model.objVal:,.2f}")
print(f"Number of non-zero variables: {sum(1 for v in model.getVars() if v.X > 1e-6)}")

Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (mac64[rosetta2] - Darwin 24.2.0 24C2101)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 728 rows, 2338 columns and 7084 nonzeros (Max)
Model fingerprint: 0xe3689fb5
Model has 2205 linear objective coefficients
Variable types: 105 continuous, 2233 integer (238 binary)
Coefficient statistics:
  Matrix range     [1e+00, 6e+02]
  Objective range  [2e+01, 5e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [5e+00, 7e+02]
Found heuristic solution: objective 5250.0000000
Presolve removed 688 rows and 2029 columns
Presolve time: 0.03s
Presolved: 40 rows, 309 columns, 708 nonzeros
Found heuristic solution: objective 65335.653193
Variable types: 19 continuous, 290 integer (272 binary)

Root relaxation: objective 1.040474e+05, 22 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj 

In [22]:
def optimize_showtimes(model, x, r, s, g, movie_ids, days, screens, genre_columns=None, verbose=False):
    """
    Optimize the showtime scheduling model and extract results
    
    Args:
        model: Gurobi model
        x: Decision variables for showings per movie/day/screen
        r: Decision variables for realized tickets
        s: Binary variables for whether movie is scheduled
        g: Binary variables for genre representation per day (optional)
        movie_ids: List of movie IDs
        days: List of days
        screens: List of screen names
        genre_columns: List of genre names (required if g is not None)
        verbose: Whether to print status
    """
    
    model.optimize()
    
    status_code = model.Status
    if status_code == GRB.OPTIMAL:
        status = "Optimal"
    elif status_code == GRB.INFEASIBLE:
        status = "Infeasible"
    elif status_code == GRB.UNBOUNDED:
        status = "Unbounded"
    else:
        status = f"Status_{status_code}"
    
    total_revenue = model.objVal if status_code == GRB.OPTIMAL else None
    
    if verbose:
        print("Status:", status)
        print(f"Total revenue: ${total_revenue:,.2f}" if total_revenue else "N/A")
    
    # Extract schedule (movie-day-screen showings)
    schedule_rows = []
    if status_code == GRB.OPTIMAL:
        for i in movie_ids:
            for d in days:
                for j in screens:
                    val = x[i, d, j].X
                    if val > 1e-6:
                        schedule_rows.append({
                            "movie_id": i,
                            "day": d,
                            "screen": j,
                            "showings": int(round(val))
                        })
    
    schedule_df = pd.DataFrame(schedule_rows)
    
    # Extract genre representation per day (if available)
    genres_per_day = {}
    if status_code == GRB.OPTIMAL and g is not None and genre_columns is not None:
        for d in days:
            active_genres = [genre for genre in genre_columns if g[genre, d].X > 0.5]
            genres_per_day[d] = active_genres
    
    # Extract realized tickets and scheduling decisions
    realized_rows = []
    if status_code == GRB.OPTIMAL:
        for i in movie_ids:
            for d in days:
                row_data = {
                    "movie_id": i,
                    "day": d,
                    "realized_tickets": r[i, d].X,
                    "scheduled_flag": int(round(s[i, d].X))
                }
                
                # Add genres for this day if available
                if genres_per_day:
                    row_data["genres_on_day"] = ", ".join(genres_per_day.get(d, []))
                
                realized_rows.append(row_data)
    
    realized_df = pd.DataFrame(realized_rows)
    
    return {
        "status": status,
        "total_revenue": total_revenue,
        "schedule_df": schedule_df,
        "realized_df": realized_df,
        "genres_per_day": genres_per_day  # Add this as a separate output
    }


In [23]:
# Optimize and get results
results = optimize_showtimes(model, x, r, s, g, movie_ids, DAYS, SCREENS, 
                             genre_columns=genre_columns, verbose=True)

print(f"\n{'='*60}")
print("OPTIMIZATION RESULTS")
print(f"{'='*60}")
print(f"Status: {results['status']}")
print(f"Total Revenue: ${results['total_revenue']:,.2f}" if results['total_revenue'] else "N/A")
print(f"\nSchedule has {len(results['schedule_df'])} showtime slots")
print(f"Movies scheduled: {results['realized_df'][results['realized_df']['scheduled_flag'] == 1].shape[0]} movie-day combinations")

# Display genres per day
if results['genres_per_day']:
    print(f"\n{'='*60}")
    print("GENRES REPRESENTED PER DAY")
    print(f"{'='*60}")
    for day in DAYS:
        genres = results['genres_per_day'].get(day, [])
        print(f"  {day}: {len(genres)} genres - {genres}")


Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (mac64[rosetta2] - Darwin 24.2.0 24C2101)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 728 rows, 2338 columns and 7084 nonzeros (Max)
Model fingerprint: 0xe3689fb5
Model has 2205 linear objective coefficients
Variable types: 105 continuous, 2233 integer (238 binary)
Coefficient statistics:
  Matrix range     [1e+00, 6e+02]
  Objective range  [2e+01, 5e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [5e+00, 7e+02]
Presolved: 40 rows, 309 columns, 708 nonzeros

Continuing optimization...


Explored 1 nodes (22 simplex iterations) in 0.05 seconds (0.02 work units)
Most recent optimization runtime was 0.00 seconds (0.00 work units)
Thread count was 10 (of 10 available processors)

Solution count 3: 104047 65335.7 5250 

Optimal solution found (tolerance 1.00e-04)
Best objective 1.040473802085e+05, best bound 1.040473802085e+05, gap 0.0000%
Status: Optimal

In [24]:
# Analyze the schedule
print("\n" + "="*60)
print("SCHEDULE ANALYSIS")
print("="*60)

if len(results['schedule_df']) > 0:
    schedule_df = results['schedule_df']
    realized_df = results['realized_df']
    
    # Showings per movie
    print("\nShowings per movie:")
    showings_per_movie = schedule_df.groupby('movie_id')['showings'].sum().sort_values(ascending=False)
    for movie_id, count in showings_per_movie.items():
        movie_title = current[current['id'] == movie_id]['title'].values[0] if movie_id in current['id'].values else f"Movie {movie_id}"
        print(f"  {movie_title}: {count} showings")
    
    # Tickets sold per movie
    print("\nTickets sold per movie:")
    tickets_per_movie = realized_df[realized_df['scheduled_flag'] == 1].groupby('movie_id')['realized_tickets'].sum().sort_values(ascending=False)
    for movie_id, tickets in tickets_per_movie.items():
        movie_title = current[current['id'] == movie_id]['title'].values[0] if movie_id in current['id'].values else f"Movie {movie_id}"
        revenue = tickets * TICKET_PRICE
        print(f"  {movie_title}: {tickets:,.0f} tickets (${revenue:,.2f})")
    
    # Showings per day
    print("\nShowings per day:")
    showings_per_day = schedule_df.groupby('day')['showings'].sum()
    for day in DAYS:
        count = showings_per_day.get(day, 0)
        print(f"  {day}: {count} showings")
    
    # Screen utilization
    print("\nScreen utilization:")
    screens_used = schedule_df.groupby('screen')['showings'].sum().sort_values(ascending=False)
    print(f"  Screens used: {len(screens_used)} / {len(SCREENS)}")
    for screen, count in screens_used.head(10).items():
        print(f"  {screen}: {count} showings")
    
    # Genre diversity analysis using the genres_per_day from results
    if results.get('genres_per_day'):
        print("\n" + "="*60)
        print("GENRE DIVERSITY ANALYSIS")
        print("="*60)
        
        print(f"\nGenres represented each day (MIN_GENRES = {MIN_GENRES}):")
        for day in DAYS:
            genres = results['genres_per_day'].get(day, [])
            print(f"  {day}: {len(genres)} genres - {genres}")
    
    # Display sample schedule
    print("\n" + "="*60)
    print("SAMPLE SCHEDULE (first 20 entries)")
    print("="*60)
    display(schedule_df.head(20))
    
else:
    print("\n⚠️ No schedule generated - check if demand is too low or constraints are too tight")



SCHEDULE ANALYSIS

Showings per movie:
  Movie 1292: 7 showings
  Movie 1751: 7 showings
  Movie 1793: 7 showings
  Movie 2464: 7 showings
  Movie 2486: 7 showings
  Movie 2552: 7 showings
  Movie 266: 7 showings
  Movie 2686: 7 showings
  Movie 2799: 7 showings
  Movie 2876: 7 showings
  Movie 3488: 7 showings
  Movie 3664: 7 showings
  Movie 3903: 7 showings
  Movie 471: 7 showings
  Movie 978: 7 showings

Tickets sold per movie:
  Movie 3664: 1,297 tickets ($19,458.91)
  Movie 3488: 960 tickets ($14,398.94)
  Movie 978: 821 tickets ($12,314.53)
  Movie 2464: 457 tickets ($6,853.76)
  Movie 2876: 456 tickets ($6,839.93)
  Movie 2486: 443 tickets ($6,651.45)
  Movie 2799: 410 tickets ($6,148.09)
  Movie 1793: 362 tickets ($5,424.18)
  Movie 2552: 334 tickets ($5,006.74)
  Movie 471: 289 tickets ($4,334.14)
  Movie 3903: 287 tickets ($4,309.18)
  Movie 1751: 267 tickets ($3,999.80)
  Movie 266: 252 tickets ($3,787.25)
  Movie 2686: 167 tickets ($2,505.98)
  Movie 1292: 134 tickets ($2

,movie_id,day,screen,showings
0,978,Mon,Screen_19,1
1,978,Tue,Screen_19,1
2,978,Wed,Screen_19,1
3,978,Thu,Screen_19,1
4,978,Fri,Screen_19,1
5,978,Sat,Screen_3,1
6,978,Sun,Screen_15,1
7,3664,Mon,Screen_4,1
8,3664,Tue,Screen_3,1
9,3664,Wed,Screen_3,1


## Export Results for Sensitivity Analysis

In [25]:
# Create results directory if it doesn't exist
results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

# Export schedule
schedule_file = results_dir / "baseline_schedule.csv"
results['schedule_df'].to_csv(schedule_file, index=False)
print(f"✓ Schedule exported to: {schedule_file}")

# Export realized tickets/revenue
realized_file = results_dir / "baseline_realized.csv"
results['realized_df'].to_csv(realized_file, index=False)
print(f"✓ Realized tickets exported to: {realized_file}")

# Export summary statistics
summary_data = {
    'metric': [
        'total_revenue',
        'total_showings',
        'total_tickets_sold',
        'num_movies_scheduled',
        'num_screens_used',
        'avg_showings_per_movie',
        'avg_tickets_per_showing',
        'capacity_utilization'
    ],
    'value': [
        results['total_revenue'],
        len(results['schedule_df']),
        results['realized_df']['realized_tickets'].sum(),
        results['realized_df']['scheduled_flag'].sum(),
        results['schedule_df']['screen'].nunique(),
        results['schedule_df'].groupby('movie_id')['showings'].sum().mean(),
        results['realized_df']['realized_tickets'].sum() / results['schedule_df']['showings'].sum() if len(results['schedule_df']) > 0 else 0,
        results['realized_df']['realized_tickets'].sum() / (results['schedule_df']['showings'].sum() * 210) if len(results['schedule_df']) > 0 else 0  # assuming avg capacity 210
    ]
}
summary_df = pd.DataFrame(summary_data)
summary_file = results_dir / "baseline_summary.csv"
summary_df.to_csv(summary_file, index=False)
print(f"✓ Summary statistics exported to: {summary_file}")

# Export parameters used
params_data = {
    'parameter': [
        'ticket_price',
        'buffer_min',
        'max_showings_per_movie_per_day',
        'num_screens',
        'operating_hours_per_day',
        'num_movies',
        'num_days'
    ],
    'value': [
        TICKET_PRICE,
        BUFFER_MIN,
        MAX_SHOWINGS_PER_MOVIE_PER_DAY,
        NUM_SCREENS,
        OPERATING_MIN_PER_DAY['Mon'] / 60,  # convert to hours
        len(movie_ids),
        len(DAYS)
    ]
}
params_df = pd.DataFrame(params_data)
params_file = results_dir / "baseline_parameters.csv"
params_df.to_csv(params_file, index=False)
print(f"✓ Parameters exported to: {params_file}")

# Export movie data
movie_export = current[['title', 'runtime', 'expected_demand']].copy()
movie_export['movie_id'] = current.index
movie_file = results_dir / "movie_data.csv"
movie_export.to_csv(movie_file, index=False)
print(f"✓ Movie data exported to: {movie_file}")

# Export demand data (for sensitivity analysis)
demand_data = []
for (movie_id, day), demand_val in demand.items():
    demand_data.append({
        'movie_id': movie_id,
        'day': day,
        'demand': demand_val
    })
demand_df = pd.DataFrame(demand_data)
demand_file = results_dir / "demand_data.csv"
demand_df.to_csv(demand_file, index=False)
print(f"✓ Demand data exported to: {demand_file}")

print(f"\n{'='*60}")
print("All baseline results exported successfully!")
print(f"{'='*60}")

✓ Schedule exported to: results/baseline_schedule.csv
✓ Realized tickets exported to: results/baseline_realized.csv
✓ Summary statistics exported to: results/baseline_summary.csv
✓ Parameters exported to: results/baseline_parameters.csv
✓ Movie data exported to: results/movie_data.csv
✓ Demand data exported to: results/demand_data.csv

All baseline results exported successfully!
